In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

In [ ]:
import boto3
from config import BUCKET_NAME

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

deleted_count = 0
for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="silver/"):
    objects = page.get("Contents", [])
    if not objects:
        continue

    keys = [{"Key": obj["Key"]} for obj in objects]
    s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
    deleted_count += len(keys)
    print(f"Deleted {len(keys)} objects...")

print(f"Done. Total deleted: {deleted_count}")

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

items = spark.read.csv(
    s3_path("bronze", "order_items", "olist_order_items_dataset.csv"),
    header=True,
    inferSchema=True
)

items.show(5)

In [ ]:
items.printSchema()

print(f"Number of records: {items.count()}")

items.show(10, truncate=False)

items.describe().show()

from pyspark.sql.functions import col, count, when

items.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in items.columns
]).show()

In [ ]:
from pyspark.sql.functions import col

duplicate_keys = (
    items
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_keys.show(truncate=False)

In [ ]:
total_rows = items.count()

distinct_rows = items.distinct().count()

print(f"Total Rows    : {total_rows}")
print(f"Distinct Rows : {distinct_rows}")
print(f"Duplicate Rows: {total_rows - distinct_rows}")

In [ ]:
from pyspark.sql.functions import col

print("Negative Prices:")
items.filter(col("price") < 0).show()

print("Negative Freight:")
items.filter(col("freight_value") < 0).show()